# Set-up Torch

In [ ]:
from mattg.device import device

device = device()

# Set-up MNIST dataset

In [ ]:
from mattg.datasets.bmnist import binarised_mnist

mnist = binarised_mnist()

# Visualise some MNIST samples

In [ ]:
from mattg.plotting import plot

plot(mnist)

# Set-up our (autoregressive) model

In general, we want to estimate the joint probability density function
$$
p(x_1, \ldots, x_n).
$$

The chain rule for probability gives
$$
p(x_1, \ldots, x_{n}) = p(x_1) \, p(x_2 \,|\, x_1) \,\cdots\, p(x_n \,|\, x_{n-1}, \ldots, x_1).
$$

The above problem is intractible, so we need to make some assumptions.

Let's assume a raster-scan ordering of our variables from top-left $X_1$ to bottom-right $X_{n=784}$.

Further, we assume a parameterisation of each conditional probability

$$
p(x_1, \ldots, x_{n}) = p_{\text{CPT}}(x_1; \alpha_1) \, p_{\text{logit}}(x_2 \,|\, x_1; \alpha_2) \,\cdots\, p_{\text{logit}}(x_n \,|\, x_{n-1}, \ldots, x_1; \alpha_n),
$$
where
$$
P_{\text{CPT}}(X_1 = 1; \alpha_1) = \alpha_1, \; P_{\text{CPT}}(X_1 = 0; \alpha_1) = 1 - \alpha_1,
$$
$$
P_{\text{logit}}(X_2 = 1 \,|\, x_1; \alpha_2) = \sigma(\alpha_0^2 + \alpha_1^2x_1)
$$
and $\sigma$ is the softmax function.

This is a Fully Visible Sigmoid Belief Network (FVSBN).

In [ ]:
from mattg.models.made import MADE, make_generator, make_made_masks

layer_sizes = [28*28, 28*28//2, 28*28//2, 28*28]
generator = make_generator(seed=42)
model_kwargs = {"layer_sizes": layer_sizes, "masks": make_made_masks(layer_sizes, generator)}
model = MADE(**model_kwargs).to(device)

# Training

# Training a single class

Let's train a single digit

In [ ]:
import torch
from torch.utils.data import random_split

train_data = mnist

#DIGIT = 3
#indices = [i for i, (_, y) in enumerate(mnist) if y == DIGIT]
#train_data = Subset(mnist, indices)

train_size = int(0.9 * len(train_data))
validation_size = len(train_data) - train_size
split_generator = torch.Generator().manual_seed(42)
train_data, validation_data = random_split(
    train_data, [train_size, validation_size], generator=split_generator
)

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from mattg.sampling.autoregressive import ancestral_sample


train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=64, shuffle=False)
criterion = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr=1e-3)

def _update_masks():
    model.update_masks(make_made_masks(layer_sizes, generator))

NLL_COMPUTE_PERIOD = 3
SAMPLE_COMPUTE_PERIOD = 3


@dataclass
class NLLResult:
    iteration: int
    label: int
    nlls: List[float]


@dataclass
class SampleResult:
    iteration: int
    samples: torch.Tensor
    probabilities: torch.Tensor


def _train_step() -> Tuple[float, int]:
    running_loss = 0.0
    n_examples = 0
    for x, _ in train_loader:
        x = x.to(device).view(x.size(0), -1)

        logits = model(x)
        loss = criterion(logits, x)

        optim.zero_grad()
        loss.backward()
        optim.step()

        running_loss += loss.item() * x.size(0)
        n_examples += x.size(0)
    return running_loss, n_examples


@torch.no_grad()
def _compute_nll(iteration, dataset) -> List[NLLResult]:
    model.eval()

    running_nll = 0.0
    n_examples = 0

    for x, _ in dataset:
        x = x.to(device).view(x.size(0), -1)

        logits = model(x)
        batch_nll = F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1)

        running_nll += batch_nll.sum().item()
        n_examples += x.size(0)

    avg_nll = running_nll / n_examples
    return [NLLResult(iteration=iteration, label=-1, nlls=[avg_nll])]


def _generate_samples(iteration, model) -> List[SampleResult]:
    result = ancestral_sample(model, device=device, n_samples=64)
    return [SampleResult(iteration=iteration, samples=result.samples, probabilities=result.probabilities)]


n_epochs = 90
mask_update_freq = 30
train_losses = []
train_nlls: List[NLLResult] = []
validation_nlls: List[NLLResult] = []
samples: List[SampleResult] = []

for epoch in range(n_epochs):
    model.train()
    running_loss, n_examples = _train_step()

    if (epoch + 1) % NLL_COMPUTE_PERIOD == 0:
        train_nlls.extend(_compute_nll(epoch + 1, train_loader))
        validation_nlls.extend(_compute_nll(epoch + 1, validation_loader))

    if (epoch + 1) % SAMPLE_COMPUTE_PERIOD == 0:
        samples.extend(_generate_samples(epoch + 1, model))

    epoch_loss = running_loss / n_examples
    train_losses.append(epoch_loss)
    print(f"epoch {epoch + 1}/{n_epochs} loss={epoch_loss:.4f}")

    if (epoch+1) % mask_update_freq == 0 and (epoch+1) < n_epochs:
        print("Updating MADE masks")
        _update_masks()

train_nlls.extend(_compute_nll(n_epochs, train_loader))
validation_nlls.extend(_compute_nll(n_epochs, validation_loader))
samples.extend(_generate_samples(n_epochs, model))

# Generate some samples

# Training loss

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

train_epochs = [entry.iteration for entry in train_nlls]
train_values = [entry.nlls[0] for entry in train_nlls]
validation_epochs = [entry.iteration for entry in validation_nlls]
validation_values = [entry.nlls[0] for entry in validation_nlls]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_epochs, train_values, marker="o", label="train NLL")
ax.plot(validation_epochs, validation_values, marker="o", label="validation NLL")
ax.set_xlabel("epoch")
ax.set_ylabel("NLL")
ax.set_title("NLL over epochs")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "made_nll_over_epochs.png", dpi=160)
plt.close(fig)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

fig_dir = Path("figures")
sample_dir = fig_dir / "samples"
sample_dir.mkdir(parents=True, exist_ok=True)

for sample_result in samples:
    fig, axes = plt.subplots(8, 8, figsize=(12, 12))
    for ax, img, probs in zip(axes.flat, sample_result.samples, sample_result.probabilities):
        left = img.detach().cpu().view(28, 28)
        right = probs.detach().cpu().view(28, 28)
        panel = torch.cat([left, right], dim=1)

        ax.imshow(panel, cmap="gray", interpolation="nearest")
        ax.axis("off")

    fig.suptitle(f"Samples at epoch {sample_result.iteration}")
    fig.tight_layout()
    fig.savefig(sample_dir / f"made_samples_epoch_{sample_result.iteration:03d}.png", dpi=160)
    plt.close(fig)

In [ ]:
from mattg.models.io import save_model

save_model(model, "made.pt", model_name="MADE", init_kwargs=model_kwargs)